# Deepfake Detection Training Pipeline (PyTorch + ConvNeXt + 2D FFT)
**Hardware Target**: Kaggle / Colab 2x NVIDIA T4 GPUs (`DataParallel` + Mixed Precision `fp16`)
**Key Architectural Features**:
- Group-based **Video-ID Partitioning** (Eliminates frame-level data leakage)
- Dynamic **Relative Bounding Box Scale Padding** (1.30x expansion)
- Dual-Stream **ConvNeXt-Small + 2D FFT Frequency** Architecture
- Centralized **YAML Configuration Management** (`src/config.py`)
- Quantitative **Frequency Stream Ablation Study**
- **Held-Out Manipulation-Type Generalization (Leave-One-Type-Out / LOTO)**
- **ONNX Model Export** for CPU/GPU inference acceleration

> **GPU Batched MTCNN**: Cell 2 uses GPU Tensor Batching for accelerated face extraction.

In [ ]:
!pip install timm facenet-pytorch albumentations grad-cam onnx onnxruntime pyyaml -q

import os, sys, re, random, time, cv2, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import timm
from facenet_pytorch import MTCNN
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 1. Global Setup & Centralized YAML Configuration

In [ ]:
try:
    from src.config import load_config
    CFG = load_config()
except ImportError:
    CFG = {
        'paths': {'kaggle_input': '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23', 'output_dir': '/kaggle/working/frames_v2'},
        'preprocessing': {'img_size': 224, 'padding_scale': 1.30, 'max_videos_per_type': 100, 'frames_per_video': 15},
        'training': {'batch_size': 64, 'epochs_phase1': 8, 'epochs_phase2': 12, 'lr_phase1': 3e-4, 'seed': 42},
        'manipulation_types': {'all': ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter', 'DeepFakeDetection'], 'held_out_loto': 'FaceShifter'}
    }

SEED = CFG.get('training', {}).get('seed', 42)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = CFG.get('paths', {}).get('kaggle_input', '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23')
OUTPUT_DIR = "/kaggle/working/frames_v2"
IMG_SIZE = CFG.get('preprocessing', {}).get('img_size', 224)
PADDING_SCALE = CFG.get('preprocessing', {}).get('padding_scale', 1.30)
FAKE_DIRS = CFG.get('manipulation_types', {}).get('all', ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter', 'DeepFakeDetection'])
HELD_OUT_TYPE = CFG.get('manipulation_types', {}).get('held_out_loto', 'FaceShifter')

os.makedirs(f"{OUTPUT_DIR}/real", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/fake", exist_ok=True)
print("Global configuration loaded successfully.")

## 2. Batched Face Extraction

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
mtcnn = MTCNN(keep_all=True, post_process=False, device=device, select_largest=True)

def extract_faces_batched(folder_path, label_str, max_videos=100, frames_per_video=15):
    folder_name = os.path.basename(folder_path)
    out_dir = f"{OUTPUT_DIR}/{label_str}"
    if not os.path.exists(folder_path):
        return
        
    videos = [f for f in os.listdir(folder_path) if f.endswith('.mp4')][:max_videos]
    saved = 0
    
    for v in tqdm(videos, desc=f"Extracting {folder_name}"):
        v_path = os.path.join(folder_path, v)
        cap = cv2.VideoCapture(v_path)
        if not cap.isOpened():
            continue
            
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            continue
            
        actual = min(frames_per_video, total)
        step = max(total // actual, 1)
        target_frames = set(i * step for i in range(actual))
        v_name = os.path.splitext(v)[0]
        
        frames_pil, frames_rgb = [], []
        curr_frame = 0
        max_target = max(target_frames)
        
        while cap.isOpened() and len(frames_rgb) < actual and curr_frame <= max_target:
            if curr_frame in target_frames:
                ret, frame = cap.read()
                if ret and frame is not None:
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames_rgb.append(rgb)
                    frames_pil.append(Image.fromarray(rgb))
            else:
                cap.grab()
            curr_frame += 1
        cap.release()
        
        if not frames_pil:
            continue
            
        try:
            boxes_list, _ = mtcnn.detect(frames_pil)
            if boxes_list is None:
                boxes_list = [None] * len(frames_pil)
        except Exception:
            boxes_list = [None] * len(frames_pil)
            
        for idx, (rgb, boxes) in enumerate(zip(frames_rgb, boxes_list)):
            if boxes is None or len(boxes) == 0:
                continue
                
            h, w, _ = rgb.shape
            best_box = max(boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]))
            x1, y1, x2, y2 = best_box[:4]
            bw, bh = x2 - x1, y2 - y1
            if bw < 10 or bh < 10:
                continue
                
            cx, cy = (x1 + x2)/2.0, (y1 + y2)/2.0
            nbw, nbh = bw * PADDING_SCALE, bh * PADDING_SCALE
            nx1 = max(0, int(cx - nbw/2.0))
            ny1 = max(0, int(cy - nbh/2.0))
            nx2 = min(w, int(cx + nbw/2.0))
            ny2 = min(h, int(cy + nbh/2.0))
            
            face = rgb[ny1:ny2, nx1:nx2]
            if face.size == 0 or face.shape[0] < 10 or face.shape[1] < 10:
                continue
                
            face_resized = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
            fname = f"{folder_name}_{v_name}_f{idx}.png"
            cv2.imwrite(os.path.join(out_dir, fname), cv2.cvtColor(face_resized, cv2.COLOR_RGB2BGR))
            saved += 1
            
    print(f"Extraction complete for {folder_name}: {saved} face crops stored.")

if os.path.exists(BASE):
    extract_faces_batched(os.path.join(BASE, "original"), "real", max_videos=100, frames_per_video=15)
    for fd in FAKE_DIRS:
        extract_faces_batched(os.path.join(BASE, fd), "fake", max_videos=100, frames_per_video=15)

## 3. Video-ID GroupKFold Data Partitioning

In [ ]:
def extract_video_id(fname):
    base = os.path.splitext(fname)[0]
    base_no_frame = re.sub(r'_f\d+$', '', base)
    match_pair = re.search(r'(\d+)_\d+', base_no_frame)
    if match_pair:
        return match_pair.group(1)
    match_single = re.search(r'(\d+)', base_no_frame)
    if match_single:
        return match_single.group(1)
    return base_no_frame.split('_')[0]

def perform_group_split(files, label, test_size=0.15, val_size=0.15):
    video_map = {}
    for f in files:
        vid = extract_video_id(os.path.basename(f))
        if vid not in video_map:
            video_map[vid] = []
        video_map[vid].append(f)
        
    unique_vids = list(video_map.keys())
    random.shuffle(unique_vids)
    
    n_total = len(unique_vids)
    n_test = max(1, int(n_total * test_size))
    n_val = max(1, int(n_total * val_size))
    
    test_vids = set(unique_vids[:n_test])
    val_vids = set(unique_vids[n_test:n_test + n_val])
    train_vids = set(unique_vids[n_test + n_val:])
    
    assert len(train_vids.intersection(val_vids)) == 0
    assert len(train_vids.intersection(test_vids)) == 0
    assert len(val_vids.intersection(test_vids)) == 0
    
    train_f = [(f, label) for vid in train_vids for f in video_map[vid]]
    val_f = [(f, label) for vid in val_vids for f in video_map[vid]]
    test_f = [(f, label) for vid in test_vids for f in video_map[vid]]
    
    return train_f, val_f, test_f

real_files = [os.path.join(OUTPUT_DIR, "real", f) for f in os.listdir(f"{OUTPUT_DIR}/real")] if os.path.exists(f"{OUTPUT_DIR}/real") else []
fake_files = [os.path.join(OUTPUT_DIR, "fake", f) for f in os.listdir(f"{OUTPUT_DIR}/fake")] if os.path.exists(f"{OUTPUT_DIR}/fake") else []

print(f"Total Real Files: {len(real_files)} | Total Fake Files: {len(fake_files)}")
tr_r, val_r, ts_r = perform_group_split(real_files, 1)
tr_f, val_f, ts_f = perform_group_split(fake_files, 0)

train_samples = tr_r + tr_f
val_samples = val_r + val_f
test_samples = ts_r + ts_f
random.shuffle(train_samples)

print(f"Train samples: {len(train_samples)} | Val samples: {len(val_samples)} | Test samples: {len(test_samples)}")

## 4. PyTorch Dataset & Albumentations Augmentations

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.4),
    A.ImageCompression(quality_range=(60, 100), p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

class DeepfakeDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if bgr is not None:
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        else:
            with Image.open(path) as pil_img:
                img = np.array(pil_img.convert("RGB"))
        if self.transform:
            img = self.transform(image=img)['image']
        else:
            img = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0
            img = (img - torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)) / torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        return img, torch.tensor(label, dtype=torch.float32)

batch_sz = CFG.get('training', {}).get('batch_size', 64)
train_ds = DeepfakeDataset(train_samples, train_transform)
val_ds = DeepfakeDataset(val_samples, eval_transform)
test_ds = DeepfakeDataset(test_samples, eval_transform)

train_loader = DataLoader(train_ds, batch_size=batch_sz, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_sz, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_sz, shuffle=False, num_workers=4, pin_memory=True)

## 5. Dual-Stream PyTorch Model Architecture (ConvNeXt + 2D FFT)

In [ ]:
class FFTFrequencyExtractor(nn.Module):
    def __init__(self, out_features=128):
        super().__init__()
        self.rgb_to_gray = nn.Conv2d(3, 1, kernel_size=1, bias=False)
        with torch.no_grad():
            self.rgb_to_gray.weight.data = torch.tensor([[[[0.299]], [[0.587]], [[0.114]]]], dtype=torch.float32)
        self.rgb_to_gray.weight.requires_grad = False
        
        self.conv_net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        self.fc = nn.Linear(128, out_features)
        
    def forward(self, x):
        gray = self.rgb_to_gray(x).to(torch.float32)
        fft = torch.fft.fft2(gray)
        fft_shift = torch.fft.fftshift(fft, dim=(-2, -1))
        magnitude = torch.abs(fft_shift)
        log_spectrum = torch.log(magnitude + 1e-5)
        
        flat_spectrum = log_spectrum.flatten(1)
        min_val = flat_spectrum.min(dim=1, keepdim=True)[0].unsqueeze(-1).unsqueeze(-1)
        max_val = flat_spectrum.max(dim=1, keepdim=True)[0].unsqueeze(-1).unsqueeze(-1)
        norm_spectrum = (log_spectrum - min_val) / (max_val - min_val + 1e-5)
        norm_spectrum = norm_spectrum.to(x.dtype)
        
        feat = self.conv_net(norm_spectrum)
        return self.fc(feat)

class HybridDeepfakeDetector(nn.Module):
    def __init__(self, backbone_name="convnext_small", pretrained=True, use_fft_branch=True, dropout=0.3):
        super().__init__()
        self.use_fft_branch = use_fft_branch
        self.spatial_backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        spatial_in_features = self.spatial_backbone.num_features
        
        if self.use_fft_branch:
            self.freq_extractor = FFTFrequencyExtractor(out_features=128)
            fusion_dim = spatial_in_features + 128
        else:
            self.freq_extractor = None
            fusion_dim = spatial_in_features
            
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
        
    def forward(self, x):
        spatial_feat = self.spatial_backbone(x)
        if self.use_fft_branch and self.freq_extractor is not None:
            freq_feat = self.freq_extractor(x)
            fused = torch.cat([spatial_feat, freq_feat], dim=1)
        else:
            fused = spatial_feat
        logits = self.classifier(fused)
        return logits.squeeze(-1)

def build_model(use_fft=True):
    model = HybridDeepfakeDetector(backbone_name="convnext_small", pretrained=True, use_fft_branch=use_fft)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    return model.to(device)

## 6. Multi-GPU Training Loop with AMP (fp16)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=3e-4):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler()
    
    best_val_auc = 0.0
    best_weights = None
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                logits = model(imgs)
                loss = criterion(logits, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * imgs.size(0)
            
        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_probs, val_targets = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                with torch.cuda.amp.autocast():
                    logits = model(imgs)
                    probs = torch.sigmoid(logits)
                val_probs.extend(probs.cpu().numpy())
                val_targets.extend(labels.numpy())
                
        val_auc = roc_auc_score(val_targets, val_probs)
        val_acc = np.mean((np.array(val_probs) > 0.5) == np.array(val_targets))
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val AUC: {val_auc:.4f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_weights = model.state_dict()
            
    model.load_state_dict(best_weights)
    return model

print("Training helper function initialized.")

## 7. Train & Evaluate Dual-Stream Model

In [ ]:
epochs_p1 = CFG.get('training', {}).get('epochs_phase1', 8)
lr_p1 = CFG.get('training', {}).get('lr_phase1', 3e-4)
model_dual = build_model(use_fft=True)
print("Training Dual-Stream Model (ConvNeXt + 2D FFT)...")
model_dual = train_model(model_dual, train_loader, val_loader, epochs=epochs_p1, lr=lr_p1)

# Evaluate on Test Set
model_dual.eval()
test_probs, test_targets = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            probs = torch.sigmoid(model_dual(imgs))
        test_probs.extend(probs.cpu().numpy())
        test_targets.extend(labels.numpy())
        
test_preds = (np.array(test_probs) > 0.5).astype(int)
print("\nClassification Report (Dual-Stream Model):")
print(classification_report(test_targets, test_preds, target_names=["Fake", "Real"]))
print(f"Test AUC: {roc_auc_score(test_targets, test_probs):.4f}")

## 8. Quantitative Frequency-Branch Ablation Study

In [ ]:
print("--- ABLATION STUDY: Spatial-Only (No FFT) vs Dual-Stream (Spatial + FFT) ---")
model_spatial = build_model(use_fft=False)
model_spatial = train_model(model_spatial, train_loader, val_loader, epochs=epochs_p1, lr=lr_p1)

model_spatial.eval()
spatial_probs = []
with torch.no_grad():
    for imgs, _ in test_loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            probs = torch.sigmoid(model_spatial(imgs))
        spatial_probs.extend(probs.cpu().numpy())
        
spatial_auc = roc_auc_score(test_targets, spatial_probs)
spatial_acc = np.mean((np.array(spatial_probs) > 0.5) == np.array(test_targets))
dual_auc = roc_auc_score(test_targets, test_probs)
dual_acc = np.mean((np.array(test_probs) > 0.5) == np.array(test_targets))

print("\n=================== ABLATION RESULTS ===================")
print(f"Spatial-Only (ConvNeXt):      Acc = {spatial_acc*100:.2f}% | AUC = {spatial_auc:.4f}")
print(f"Dual-Stream (ConvNeXt + FFT): Acc = {dual_acc*100:.2f}% | AUC = {dual_auc:.4f}")
print(f"Delta (FFT Improvement):      Acc = +{(dual_acc - spatial_acc)*100:.2f}% | AUC = +{dual_auc - spatial_auc:.4f}")
print("========================================================")

## 9. Held-Out Manipulation-Type Generalization (LOTO Benchmark)

In [ ]:
print(f"--- HELD-OUT MANIPULATION TEST (LOTO): Holding out '{HELD_OUT_TYPE}' ---")
loto_train_samples = [s for s in train_samples if HELD_OUT_TYPE.lower() not in s[0].lower()]
loto_val_samples = [s for s in val_samples if HELD_OUT_TYPE.lower() not in s[0].lower()]
loto_test_samples = [s for s in fake_files if HELD_OUT_TYPE.lower() in s.lower()]
loto_test_samples = [(f, 0) for f in loto_test_samples] + ts_r

print(f"LOTO Train samples (excluding {HELD_OUT_TYPE}): {len(loto_train_samples)}")
print(f"LOTO Val samples (excluding {HELD_OUT_TYPE}): {len(loto_val_samples)}")
print(f"LOTO Test samples (only {HELD_OUT_TYPE} + Real): {len(loto_test_samples)}")

loto_train_loader = DataLoader(DeepfakeDataset(loto_train_samples, train_transform), batch_size=batch_sz, shuffle=True, num_workers=4)
loto_val_loader = DataLoader(DeepfakeDataset(loto_val_samples, eval_transform), batch_size=batch_sz, shuffle=False, num_workers=4)
loto_test_loader = DataLoader(DeepfakeDataset(loto_test_samples, eval_transform), batch_size=batch_sz, shuffle=False, num_workers=4)

model_loto = build_model(use_fft=True)
model_loto = train_model(model_loto, loto_train_loader, loto_val_loader, epochs=epochs_p1, lr=lr_p1)

model_loto.eval()
loto_probs, loto_targets = [], []
with torch.no_grad():
    for imgs, labels in loto_test_loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            probs = torch.sigmoid(model_loto(imgs))
        loto_probs.extend(probs.cpu().numpy())
        loto_targets.extend(labels.numpy())
        
loto_auc = roc_auc_score(loto_targets, loto_probs)
loto_acc = np.mean((np.array(loto_probs) > 0.5) == np.array(loto_targets))
print(f"\nUnseen Manipulation '{HELD_OUT_TYPE}' Generalization Accuracy: {loto_acc*100:.2f}% | AUC: {loto_auc:.4f}")

## 10. PyTorch & ONNX Model Export

In [ ]:
# 1. Save PyTorch State Dict (.pth)
export_pth_path = "/kaggle/working/deepfake_convnext_v2.pth"
unwrapped_model = model_dual.module if hasattr(model_dual, 'module') else model_dual
torch.save(unwrapped_model.state_dict(), export_pth_path)
print(f"PyTorch model weights saved to {export_pth_path}")

# 2. Export ONNX Model (.onnx) for inference acceleration
try:
    import onnx
    export_onnx_path = "/kaggle/working/deepfake_convnext_v2.onnx"
    unwrapped_model.eval()
    dummy_in = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    torch.onnx.export(
        unwrapped_model,
        dummy_in,
        export_onnx_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    print(f"ONNX model successfully exported to {export_onnx_path}")
except Exception as e:
    print(f"ONNX export failed: {e}")